Compute model evidence, P(D|M), for all models developped in this project.

FEV1, 2-day FEV1 FEF, long model

In [1]:
import concurrent.futures
from itertools import repeat

import numpy as np
import pandas as pd
import models.builders as mb
from pgmpy.inference.ExactInference import VariableElimination
from scipy.stats import wilcoxon
from scipy.stats import chi2

import data.breathe_data as bd
import data.helpers as dh
import inf_cutset_conditioning.cutset_cond_algs_learn_ar_change as cca_ar_change
import inf_cutset_conditioning.cutset_cond_algs_learn_ar_change_noo2sat as cca_ar_change_noo2sat
import model_validation.model_evidence as me

In [2]:
df = bd.load_meas_from_excel("BR_O2_FEV1_FEF2575_conservative_smoothing_with_idx")

INFO:root:* Checking for same day measurements *


# FEV1 model evidence

In [3]:
# P(FEV1|M)

df_rmax_rows = (
    df.sort_values(by=["FEV1", "FEF2575", "O2 Saturation"], ascending=False)
    .groupby("ID")
    .agg(lambda df: df.head(3).tail(1))
    .reset_index()
)


def run_ve(df, with_fef2575=False, with_rmax=False):
    """
    Last measurement on row 1
    Robust max FEV1 on row 2
    """
    df = df.reset_index()

    id, height, age, sex = df.iloc[0][["ID", "Height", "Age", "Sex"]]
    # ar_prior = "breathe (2 days model, ecFEV1 addmultnoise, ecFEF25-75)"
    ar_prior = "uniform"
    ecfev1_noise_model_suffix = "_std_add_mult_ecfev1"
    fef2575_cpt_suffix = "_ecfev1_2_days_model_add_mult_noise"

    (
        model,
        HFEV1,
        AR_vars,
        uFEV1_vars,
        ecFEV1_vars,
        ecFEF2575prctecFEV1_vars,
    ) = mb.fev1_fef2575_n_day_BN_noise(
        2 if with_rmax else 1,
        height,
        age,
        sex,
        ar_prior,
        fef2575_cpt_suffix,
        ecfev1_noise_model_suffix,
    )
    var_elim = VariableElimination(model)

    evidence_dict = {}
    if with_fef2575:
        evidence_dict[ecFEF2575prctecFEV1_vars[0].name] = df.loc[
            0, "idx ecFEF2575%ecFEV1"
        ]
    if with_rmax:
        [rmax_fev1] = df_rmax_rows[df_rmax_rows.ID == id]["idx ecFEV1 (L)"].values
        evidence_dict[ecFEV1_vars[1].name] = rmax_fev1
    if with_rmax and with_fef2575:
        [rmax_fef2575] = df_rmax_rows[df_rmax_rows.ID == id][
            "idx ecFEF2575%ecFEV1"
        ].values
        evidence_dict[ecFEF2575prctecFEV1_vars[1].name] = rmax_fef2575

    # print(f"n days: {len(AR_vars)}, evidence_dict: {evidence_dict}")

    res_ve = var_elim.query(
        variables=[ecFEV1_vars[0].name],
        evidence=evidence_dict,
        joint=False,
    )
    dist_ecfev1_ve = res_ve[ecFEV1_vars[0].name].values
    p_ecfev1_ve = dist_ecfev1_ve[df.loc[0, "idx ecFEV1 (L)"]]
    return p_ecfev1_ve

In [4]:
s1 = df.groupby("ID").apply(
    lambda dftmp: run_ve(dftmp.head(1), with_fef2575=False, with_rmax=False)
)
s2 = df.groupby("ID").apply(
    lambda dftmp: run_ve(dftmp.head(1), with_fef2575=False, with_rmax=True)
)
s3 = df.groupby("ID").apply(
    lambda dftmp: run_ve(dftmp.head(1), with_fef2575=True, with_rmax=False)
)
s4 = df.groupby("ID").apply(
    lambda dftmp: run_ve(dftmp.head(1), with_fef2575=True, with_rmax=True)
)

df1 = pd.DataFrame(s1, columns=["P(FEV1|M)"]).reset_index()
df2 = pd.DataFrame(s2, columns=["P(FEV1|M, rmax FEV1)"]).reset_index()
df3 = pd.DataFrame(s3, columns=["P(FEV1|M, FEF2575)"]).reset_index()
df4 = pd.DataFrame(s4, columns=["P(FEV1|M, rmax FEV1, FEF2575)"]).reset_index()
df_p = df1.merge(df2, on="ID")
df_p = df_p.merge(df3, on="ID")
df_p = df_p.merge(df4, on="ID")

In [17]:
# Check test assumptions

diff12 = np.log(s2 - s1)
diff13 = np.log(s3 - s1)
diff23 = np.log(s3 - s2)
diff14 = np.log(s4 - s1)
diff24 = np.log(s4 - s2)
diff34 = np.log(s4 - s3)

# Symmetry:
for diff in [diff12, diff13, diff23, diff14, diff24, diff34]:
    print(diff.mean())
    print(diff.median())
    print(diff.skew())
    print(len(diff))
    print()

-7.681824630052143
-8.070600241859925
0.3778300251612568
352

-4.984115594603068
-4.768631954618282
-2.589568922359935
352

-5.016433681997174
-4.857404702255337
-2.1328739543910684
352

-4.716542872113071
-4.626972137120584
-1.656163870474271
352

-4.804702179900191
-4.669516537981096
-2.0616036940544924
352

-6.724741153153948
-6.877345570712388
-0.12580706686340903
352



/Applications/anaconda3/envs/phd/lib/python3.10/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [15]:
import numpy as np
import scipy.stats as st

# x, y are numpy arrays of your log-transformed observations
# n ~ 352 each
x= np.log(s1)
y = np.log(s2)

# 1) Welch t-test
tstat, pval = st.ttest_ind(x, y, equal_var=False)
stat_12, pval_12  = wilcoxon(y, x, alternative="greater")
print("Welch t:", tstat, "p =", pval)
print("Wilcoxon:", stat_12, "p =", pval_12)

# 2) Bootstrap 95% CI for difference in means (10000 resamples)
def bootstrap_diff_means(x, y, n_boot=10000):
    rng = np.random.default_rng()
    diffs = np.empty(n_boot)
    nx, ny = len(x), len(y)
    for i in range(n_boot):
        bx = rng.choice(x, size=nx, replace=True)
        by = rng.choice(y, size=ny, replace=True)
        diffs[i] = bx.mean() - by.mean()
    return np.percentile(diffs, [2.5, 97.5]), diffs

ci, diffs = bootstrap_diff_means(x, y)
print("Bootstrap 95% CI for mean difference (log-scale):", ci)

# 3) Permutation test for difference in means (5000 perms)
def perm_test_mean(x, y, n_perm=5000):
    rng = np.random.default_rng()
    obs = x.mean() - y.mean()
    pooled = np.concatenate([x, y])
    count = 0
    n = len(x)
    for _ in range(n_perm):
        rng.shuffle(pooled)
        if pooled[:n].mean() - pooled[n:].mean() >= abs(obs):  # two-sided
            count += 1
    p_emp = (count + 1) / (n_perm + 1)
    return obs, p_emp

obs_diff, p_perm = perm_test_mean(x, y)
print("Observed diff (log-scale):", obs_diff, "Permutation p (approx):", p_perm)

# 4) Convert log-difference to ratio (original scale)
log_diff = x.mean() - y.mean()
ratio = np.exp(log_diff)
print("Estimated geometric mean ratio (group x / group y):", ratio)


Welch t: -2.957530656286629 p = 0.003216165703297503
Wilcoxon: 57114.0 p = 1.2373085086042177e-42
Bootstrap 95% CI for mean difference (log-scale): [-0.20476426 -0.04273655]
Observed diff (log-scale): -0.12301737261894896 Permutation p (approx): 0.0003999200159968006
Estimated geometric mean ratio (group x / group y): 0.8842483007185988


In [16]:
# Wilcoxon positive-rank test for paired samples
# Paired comparison of log-likelihoods
stat_12, pval_12 = wilcoxon(np.log(s2), np.log(s1), alternative="greater")
stat_13, pval_13 = wilcoxon(np.log(s3), np.log(s1), alternative="greater")
stat_23, pval_23 = wilcoxon(np.log(s3), np.log(s2), alternative="greater")
stat_14, pval_14 = wilcoxon(np.log(s4), np.log(s1), alternative="greater")
stat_24, pval_24 = wilcoxon(np.log(s4), np.log(s2), alternative="greater")
stat_34, pval_34 = wilcoxon(np.log(s4), np.log(s3), alternative="greater")

m12 = (s2 - s1) / s1 * 100
m13 = (s3 - s1) / s1 * 100
m23 = (s3 - s2) / s2 * 100
m14 = (s4 - s1) / s1 * 100
m24 = (s4 - s2) / s2 * 100
m34 = (s4 - s3) / s3 * 100
tot = len(s1)


def compare(full, nested, m, pval, tot):
    print(
        f"{full} - {nested} median = +{m.median():.4f}% ({pval:.0e}), >0%: {(m>0).sum()/tot*100:.0f}% ({(m>0).sum()}), >30%: {(m>30).sum()/tot*100:.0f}% ({(m>30).sum()}), >100%: {(m>100).sum()/tot*100:.0f}% ({(m>100).sum()})"
    )


compare("P(FEV1|M, rmax FEV1)", "baseline", m12, pval_12, tot)
compare("P(FEV1|M, FEF2575)", "baseline", m13, pval_13, tot)
compare("P(FEV1|M, rmax FEV1, FEF2575)", "baseline", m14, pval_14, tot)
compare("P(FEV1|M, rmax FEV1, FEF2575)", "P(FEV1|M, rmax FEV1)", m24, pval_24, tot)
compare("P(FEV1|M, rmax FEV1, FEF2575)", "P(FEV1|M, FEF2575)", m34, pval_34, tot)

P(FEV1|M, rmax FEV1) - baseline median = +1.7249% (1e-42), >0%: 91% (322), >30%: 11% (40), >100%: 5% (19)
P(FEV1|M, FEF2575) - baseline median = +41.2075% (9e-35), >0%: 80% (282), >30%: 60% (210), >100%: 21% (75)
P(FEV1|M, rmax FEV1, FEF2575) - baseline median = +49.9343% (2e-42), >0%: 84% (294), >30%: 68% (238), >100%: 26% (91)
P(FEV1|M, rmax FEV1, FEF2575) - P(FEV1|M, rmax FEV1) median = +45.2822% (2e-40), >0%: 83% (292), >30%: 66% (231), >100%: 22% (78)
P(FEV1|M, rmax FEV1, FEF2575) - P(FEV1|M, FEF2575) median = +3.9193% (1e-41), >0%: 85% (300), >30%: 18% (65), >100%: 7% (23)


In [ ]:
# P(FEV1|M) - OLD - using cutset cond alg


def run_1d_model(df, not_fef2575=False):
    ar_prior = "breathe (2 days model, ecFEV1 addmultnoise, ecFEF25-75)"
    ar_change_cpt_suffix = "_shape_factor_single_laplace_1.6"
    ecfev1_noise_model_suffix = "_std_add_mult_ecfev1"
    fef2575_cpt_suffix = "_ecfev1_2_days_model_add_mult_noise"

    ecfef2575_cols = [
        "ecFEF2575%ecFEV1",
        "idx ecFEF2575%ecFEV1",
        "idx ecFEF25-75 % ecFEV1 (%)",
    ]

    dftmp = df.reset_index()
    if not_fef2575:
        dftmp[ecfef2575_cols] = np.nan

    ([log_p_FEV1_given_S], _) = cca_ar_change_noo2sat.run_long_noise_model_through_time(
        dftmp,
        ar_prior=ar_prior,
        ar_change_cpt_suffix=ar_change_cpt_suffix,
        ecfev1_noise_model_suffix=ecfev1_noise_model_suffix,
        fef2575_cpt_suffix=fef2575_cpt_suffix,
        get_p_fev1_given_s=True,
    )
    return log_p_FEV1_given_S

In [ ]:
# Robust average improvement via bootstrapping
def get_bootstrapped_med(s1, s2):
    n_boot = 5000
    frac = 0.1
    rng = np.random.default_rng(42)
    impr = (s2 - s1) / s1 * 100
    impr = impr.dropna()
    boot_means = []
    for _ in range(n_boot):
        sample = impr.sample(frac=frac, replace=True, random_state=rng.integers(0, 1e9))
        boot_means.append(sample.mean())
    print(np.median(boot_means))


get_bootstrapped_med(s1, s2)
get_bootstrapped_med(s1, s3)
get_bootstrapped_med(s1, s4)

17.601488968846425
60.23716232463196
115.8104332403913


In [ ]:
(((s2 - s1) / s1 * 100) > 30).sum()

# Avg relative improvement

40

In [ ]:
# Log-likelihood test
l1 = np.log(s1).sum()
l2 = np.log(s2).sum()
l3 = np.log(s3).sum()
l4 = np.log(s4).sum()

n_free_param = 4455
LR_12 = -2 * (l1 - l4) * 20  # 6000
p_value_12 = chi2.sf(LR_12, df=n_free_param)
p_value_12
# Can't use this because chi-squared approximates assumes n >> parameters. Here there are too many parameters for the model.

In [ ]:
# P(FEV1|M) - OLD - using cutset cond alg


def run_1d_model(df, not_fef2575=False):
    ar_prior = "breathe (2 days model, ecFEV1 addmultnoise, ecFEF25-75)"
    ar_change_cpt_suffix = "_shape_factor_single_laplace_1.6"
    ecfev1_noise_model_suffix = "_std_add_mult_ecfev1"
    fef2575_cpt_suffix = "_ecfev1_2_days_model_add_mult_noise"

    ecfef2575_cols = [
        "ecFEF2575%ecFEV1",
        "idx ecFEF2575%ecFEV1",
        "idx ecFEF25-75 % ecFEV1 (%)",
    ]

    dftmp = df.reset_index()
    if not_fef2575:
        dftmp[ecfef2575_cols] = np.nan

    ([log_p_FEV1_given_S], _) = cca_ar_change_noo2sat.run_long_noise_model_through_time(
        dftmp,
        ar_prior=ar_prior,
        ar_change_cpt_suffix=ar_change_cpt_suffix,
        ecfev1_noise_model_suffix=ecfev1_noise_model_suffix,
        fef2575_cpt_suffix=fef2575_cpt_suffix,
        get_p_fev1_given_s=True,
    )
    return log_p_FEV1_given_S

In [ ]:
s1 = df.groupby(["ID"]).apply(lambda dftmp: run_1d_model(dftmp.tail(1), True))
s2 = df.groupby(["ID"]).apply(lambda dftmp: run_1d_model(dftmp.tail(1), False))
df1 = pd.DataFrame(np.exp(s1), columns=["P(FEV1|M)"]).reset_index()
df2 = pd.DataFrame(np.exp(s2), columns=["P(FEV1|M, FEF2575)"]).reset_index()
df_p = df1.merge(df2, on="ID")
median1 = np.exp(s1).median()
median2 = np.exp(s2).median()

In [ ]:
median1 = np.exp(s1).median()
median2 = np.exp(s2).median()

print(f"P(FEV1|M) median = {median1:.4f}")
print(f"P(FEV1|M, FEF2575) median = {median2:.4f}")
print(f"Improvement: {(median2-median1)/median1*100:.1f}%")

P(FEV1|M) median = 0.0190
P(FEV1|M, FEF2575) median = 0.0249
Improvement: 31.0%


# FEV1 FEF2575 model evidence

In [3]:
# P(D|M) on the strict 30-day long model (excluding ID below 30 days)
# Without FEF25-75, log_p_S_given_D = -49.08053225
# With FEV1 and FEF25-75, log_p_S_given_D = -148.65322444

In [5]:
# Reduce dataset to the last 30-day sequences

ndays = 30
df30 = pd.DataFrame(columns=df.columns)
for id in df.ID.unique():
    df_pre, start_idx, end_idx = dh.find_longest_conseq_sequence(
        df[df.ID == id], n_missing_days_allowed=1
    )

    dftmp = df_pre.tail(ndays).reset_index()

    if len(dftmp) < ndays:
        # print(f"Skipping ID {id}, n entries < {ndays} days")
        continue

    df30 = pd.concat([df30, dftmp])

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_54233/3171043079.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df30 = pd.concat([df30, dftmp])


# Longitudinal model

In [ ]:
if __name__ == "__main__":
    with concurrent.futures.ProcessPoolExecutor() as executor:
        res_long_model = list(
            executor.map(me.process_id_long_model, df30.ID.unique(), repeat(df30))
            # executor.map(me.process_id_long_model, ['102'], repeat(df30))
        )

Warning - min_possible_hfev1_under_model: 2
Warning - min_possible_hfev1_under_model: 17
Warning - min_possible_hfev1_under_model: 6
123 - Time for 30 entries: 26.55 s
109 - Time for 30 entries: 29.01 s
125 - Time for 30 entries: 28.66 s
106 - Time for 30 entries: 29.85 s
101 - Time for 30 entries: 29.64 s
116 - Time for 30 entries: 30.66 s
Warning - min_possible_hfev1_under_model: 5
117 - Time for 30 entries: 32.35 s
103 - Time for 30 entries: 33.58 s
120 - Time for 30 entries: 36.00 s
111 - Time for 30 entries: 36.31 s
133 - Time for 30 entries: 25.28 s
138 - Time for 30 entries: 26.16 s
Warning - min_possible_hfev1_under_model: 25
151 - Time for 30 entries: 26.81 s
140 - Time for 30 entries: 28.05 s
146 - Time for 30 entries: 29.71 s
Warning - min_possible_hfev1_under_model: 3
147 - Time for 30 entries: 32.22 s
163 - Time for 30 entries: 26.04 s
159 - Time for 30 entries: 29.49 s
153 - Time for 30 entries: 30.81 s
162 - Time for 30 entries: 28.61 s
Warning - min_possible_hfev1_under

/Users/tristan.trebaol/Desktop/DesktopMacTristan/PhD/Code/phd/src/inf_cutset_conditioning/cutset_cond_algs.py:874: RuntimeWarning: divide by zero encountered in log
  log_p_D_given_M = np.log(p_ecFEV1)


237 - Time for 30 entries: 30.14 s
238 - Time for 30 entries: 29.24 s
250 - Time for 30 entries: 26.55 s
240 - Time for 30 entries: 33.13 s
Warning - min_possible_hfev1_under_model: 9
244 - Time for 30 entries: 32.70 s
272 - Time for 30 entries: 33.39 s
282 - Time for 30 entries: 34.00 s
Warning - min_possible_hfev1_under_model: 4
331 - Time for 30 entries: 31.76 s
Warning - min_possible_hfev1_under_model: 51
311 - Time for 30 entries: 34.39 s
Warning - min_possible_hfev1_under_model: 21
336 - Time for 30 entries: 28.51 s
352 - Time for 30 entries: 29.15 s
339 - Time for 30 entries: 29.85 s
Warning - min_possible_hfev1_under_model: 14
381 - Time for 30 entries: 27.80 s
469 - Time for 30 entries: 11.81 s
365 - Time for 30 entries: 28.20 s
405 - Time for 30 entries: 26.25 s
411 - Time for 30 entries: 27.68 s
Warning - min_possible_hfev1_under_model: 24
480 - Time for 30 entries: 19.96 s
426 - Time for 30 entries: 27.81 s
502 - Time for 30 entries: 2765.33 s
483 - Time for 30 entries: 276

In [17]:
res_long_model

[-7.38304677090481]

# 2 day FEV1, FEF25-75 model


I had to update existing implementation, made for the long model with interconnected AR. To do so, I set the AR vevidence to the first_day_prior each time. 

When fusing the models, I only took P(FEV1 day n|M, rmax FEV1, rmax FEF2575) * P(FEF2575 day n|M, rmax FEV1, rmax FEF2575, FEV1 day n). This allowed to compare the results with the other models

In [65]:
df_rmax_rows = (
    df.sort_values(by=["ecFEV1", "ecFEF2575", "O2 Saturation"], ascending=False)
    .groupby("ID")
    .agg(lambda df: df.head(1))
    .reset_index()
)
df_rmax_rows = df_rmax_rows[df_rmax_rows.ID.isin(df30.ID.unique())]

In [ ]:
df_rmax_rows[df_rmax_rows.ID == "106"]

,ID,Date Recorded,FEV1,O2 Saturation,FEF2575,ecFEV1,ecFEF2575,Sex,Height,Age,Predicted FEV1,Healthy O2 Saturation,ecFEV1 % Predicted,FEV1 % Predicted,O2 Saturation % Healthy,ecFEF2575%ecFEV1,idx ecFEV1 (L),idx O2 saturation (%),idx ecFEF2575%ecFEV1,idx ecFEF25-75 % ecFEV1 (%)
5,106,2022-08-25,1.65,96,1.22,1.65,1.22,Female,154.0,27,2.90126,98.311859,56.871845,56.871845,97.648444,73.939394,33,46,36,36


In [ ]:
if __name__ == "__main__":
    with concurrent.futures.ProcessPoolExecutor() as executor:
        res_2day_fev1_fef_model = list(
            executor.map(
                me.process_id_2day_fev1_fef_model,
                df30.ID.unique(),
                repeat(df30),
                repeat(df_rmax_rows),
            )
            # executor.map(me.process_id_2day_fev1_fef_model, ['102'], repeat(df30), repeat(df_rmax_rows))
        )

Warning - min_possible_hfev1_under_model: 4
Warning - min_possible_hfev1_under_model: 25
Warning - min_possible_hfev1_under_model: 2
Warning - min_possible_hfev1_under_model: 8


/Users/tristan.trebaol/Desktop/DesktopMacTristan/PhD/Code/phd/src/inf_cutset_conditioning/cutset_cond_algs.py:874: RuntimeWarning: divide by zero encountered in log
  log_p_D_given_M = np.log(p_ecFEV1)


123 - Time for 2 entries: 2.75 s
Warning - min_possible_hfev1_under_model: 25
125 - Time for 2 entries: 3.01 s
109 - Time for 2 entries: 3.41 s
101 - Time for 2 entries: 3.52 s
117 - Time for 2 entries: 3.40 s
103 - Time for 2 entries: 3.59 s
116 - Time for 2 entries: 3.70 s
Warning - min_possible_hfev1_under_model: 4
120 - Time for 2 entries: 3.69 s
Warning - min_possible_hfev1_under_model: 8
Warning - min_possible_hfev1_under_model: 2
106 - Time for 2 entries: 3.42 s
111 - Time for 2 entries: 3.86 s


/Users/tristan.trebaol/Desktop/DesktopMacTristan/PhD/Code/phd/src/inf_cutset_conditioning/cutset_cond_algs.py:874: RuntimeWarning: divide by zero encountered in log
  log_p_D_given_M = np.log(p_ecFEV1)


123 - Time for 2 entries: 2.49 s
Warning - min_possible_hfev1_under_model: 25
109 - Time for 2 entries: 3.17 s
125 - Time for 2 entries: 3.00 s
117 - Time for 2 entries: 3.12 s
101 - Time for 2 entries: 3.39 s
Warning - min_possible_hfev1_under_model: 4


...

480 - Time for 2 entries: 2.35 s
Warning - min_possible_hfev1_under_model: 21
517 - Time for 2 entries: 2.01 s
411 - Time for 2 entries: 3.09 s
508 - Time for 2 entries: 3.06 s
Warning - min_possible_hfev1_under_model: 1
520 - Time for 2 entries: 2.67 s
502 - Time for 2 entries: 3.07 s
506 - Time for 2 entries: 2.60 s
509 - Time for 2 entries: 3.20 s
Warning - min_possible_hfev1_under_model: 14
426 - Time for 2 entries: 2.64 s
483 - Time for 2 entries: 3.03 s
480 - Time for 2 entries: 2.36 s
Warning - min_possible_hfev1_under_model: 24
Warning - min_possible_hfev1_under_model: 2
Warning - min_possible_hfev1_under_model: 4
517 - Time for 2 entries: 2.16 s


/Users/tristan.trebaol/Desktop/DesktopMacTristan/PhD/Code/phd/src/inf_cutset_conditioning/cutset_cond_algs.py:874: RuntimeWarning: divide by zero encountered in log
  log_p_D_given_M = np.log(p_ecFEV1)


411 - Time for 2 entries: 2.93 s
508 - Time for 2 entries: 2.87 s
Warning - min_possible_hfev1_under_model: 1
506 - Time for 2 entries: 2.60 s
502 - Time for 2 entries: 2.90 s
520 - Time for 2 entries: 3.02 s
Warning - min_possible_hfev1_under_model: 14
509 - Time for 2 entries: 2.96 s
544 - Time for 2 entries: 2.20 s
Warning - min_possible_hfev1_under_model: 24
426 - Time for 2 entries: 3.03 s
483 - Time for 2 entries: 3.30 s
Warning - min_possible_hfev1_under_model: 2
Warning - min_possible_hfev1_under_model: 4
517 - Time for 2 entries: 3.01 s


/Users/tristan.trebaol/Desktop/DesktopMacTristan/PhD/Code/phd/src/inf_cutset_conditioning/cutset_cond_algs.py:874: RuntimeWarning: divide by zero encountered in log
  log_p_D_given_M = np.log(p_ecFEV1)


508 - Time for 2 entries: 3.00 s
506 - Time for 2 entries: 2.57 s
544 - Time for 2 entries: 2.31 s
Warning - min_possible_hfev1_under_model: 1
Warning - min_possible_hfev1_under_model: 24
Warning - min_possible_hfev1_under_model: 14
502 - Time for 2 entries: 2.99 s
520 - Time for 2 entries: 2.93 s
509 - Time for 2 entries: 3.14 s
483 - Time for 2 entries: 2.56 s
Warning - min_possible_hfev1_under_model: 2
426 - Time for 2 entries: 2.79 s
517 - Time for 2 entries: 2.64 s
544 - Time for 2 entries: 2.04 s
Warning - min_possible_hfev1_under_model: 24
506 - Time for 2 entries: 2.21 s
Warning - min_possible_hfev1_under_model: 14
508 - Time for 2 entries: 2.52 s
Warning - min_possible_hfev1_under_model: 1
502 - Time for 2 entries: 2.64 s
509 - Time for 2 entries: 2.61 s
520 - Time for 2 entries: 2.53 s
483 - Time for 2 entries: 2.34 s
544 - Time for 2 entries: 1.82 s
Warning - min_possible_hfev1_under_model: 2
Warning - min_possible_hfev1_under_model: 24
506 - Time for 2 entries: 2.06 s
Warni

In [15]:
res_2day_fev1_fef_model

[-207.5606173072854,
 -233.5823006389506,
 -203.83271015867493,
 -218.15198953880153,
 -229.53473057420086,
 -209.897020878688,
 -344.17205775702774,
 -248.10503101552717,
 -253.5743211428645,
 -270.86799360293486,
 -229.96840025573357,
 -209.0886061553722,
 -215.4046146307567,
 -222.35209050659722,
 -232.01776185863957,
 -233.1438234247169,
 -219.765008607971,
 -232.62850975172034,
 -205.52121175964356,
 -228.83072840513796,
 -209.36830911190825,
 -247.20060214030553,
 -243.24818413290018,
 -210.37817198666224,
 -251.90951327828856,
 -214.11068750173843,
 -198.9035209662086,
 -210.19269063565127,
 -218.2895277680156,
 -225.57217919844697,
 -226.21976937621218,
 -251.38760752650853,
 -245.9872755406521,
 -204.09531530248358,
 -218.49275065697776,
 -337.5532052916074,
 -342.8022118107003,
 -206.3148331100209,
 -210.76177404777872,
 -217.3338228574392,
 -270.5592812608371,
 -212.87494964058186,
 -214.5699772436078,
 -214.41373386769408,
 -223.04217449500106,
 -254.19812563817393,
 -226.1

In [ ]:
# with rmax FEV1, 101: -276.9759478315757
# with rmax but just taking the curr day probs, 101: -138

# FEV1, FEF25-75 model

In [ ]:
if __name__ == "__main__":
    with concurrent.futures.ProcessPoolExecutor() as executor:
        res_fev1_fef_model = list(
            executor.map(
                # me.process_id_fev1_fef_model, ['102'], repeat(df20)
                # me.process_id_fev1_fef_model, ['101'], repeat(df20)
                me.process_id_fev1_fef_model,
                df30.ID.unique(),
                repeat(df30),
            )
        )

Warning - min_possible_hfev1_under_model: 4
Warning - min_possible_hfev1_under_model: 12


/Users/tristan.trebaol/Desktop/DesktopMacTristan/PhD/Code/phd/src/inf_cutset_conditioning/cutset_cond_algs.py:874: RuntimeWarning: divide by zero encountered in log
  log_p_D_given_M = np.log(p_ecFEV1)
/Users/tristan.trebaol/Desktop/DesktopMacTristan/PhD/Code/phd/src/inf_cutset_conditioning/cutset_cond_algs.py:874: RuntimeWarning: divide by zero encountered in log
  log_p_D_given_M = np.log(p_ecFEV1)
/Users/tristan.trebaol/Desktop/DesktopMacTristan/PhD/Code/phd/src/inf_cutset_conditioning/cutset_cond_algs.py:874: RuntimeWarning: divide by zero encountered in log
  log_p_D_given_M = np.log(p_ecFEV1)


123 - Time for 1 entries: 94.41 s
117 - Time for 1 entries: 94.61 s
Warning - min_possible_hfev1_under_model: 14
111 - Time for 1 entries: 95.03 s
125 - Time for 1 entries: 94.95 s
116 - Time for 1 entries: 94.98 s
103 - Time for 1 entries: 95.06 s
109 - Time for 1 entries: 5.94 s
106 - Time for 1 entries: 6.07 s
101 - Time for 1 entries: 95.61 s
Warning - min_possible_hfev1_under_model: 4
120 - Time for 1 entries: 6.50 s
123 - Time for 1 entries: 2.34 s
117 - Time for 1 entries: 2.85 s
Warning - min_possible_hfev1_under_model: 16
111 - Time for 1 entries: 3.42 s
116 - Time for 1 entries: 3.42 s
103 - Time for 1 entries: 3.50 s
125 - Time for 1 entries: 4.09 s
106 - Time for 1 entries: 4.00 s
101 - Time for 1 entries: 4.02 s
Warning - min_possible_hfev1_under_model: 2
120 - Time for 1 entries: 3.87 s
109 - Time for 1 entries: 4.45 s
123 - Time for 1 entries: 3.00 s
117 - Time for 1 entries: 2.88 s
Warning - min_possible_hfev1_under_model: 15
111 - Time for 1 entries: 3.62 s
116 - Time 

/Users/tristan.trebaol/Desktop/DesktopMacTristan/PhD/Code/phd/src/inf_cutset_conditioning/cutset_cond_algs.py:874: RuntimeWarning: divide by zero encountered in log
  log_p_D_given_M = np.log(p_ecFEV1)


120 - Time for 1 entries: 1.88 s
123 - Time for 1 entries: 1.43 s
111 - Time for 1 entries: 2.17 s
109 - Time for 1 entries: 2.09 s
Warning - min_possible_hfev1_under_model: 16


...

Warning - min_possible_hfev1_under_model: 24
517 - Time for 1 entries: 1.11 s
509 - Time for 1 entries: 1.14 s
520 - Time for 1 entries: 1.12 s
508 - Time for 1 entries: 1.12 s
506 - Time for 1 entries: 1.01 s
Warning - min_possible_hfev1_under_model: 7
502 - Time for 1 entries: 1.18 s
544 - Time for 1 entries: 0.92 s
Warning - min_possible_hfev1_under_model: 23
509 - Time for 1 entries: 1.12 s
517 - Time for 1 entries: 1.15 s
520 - Time for 1 entries: 1.17 s
508 - Time for 1 entries: 1.11 s
544 - Time for 1 entries: 0.84 s
506 - Time for 1 entries: 1.07 s
502 - Time for 1 entries: 1.11 s
Warning - min_possible_hfev1_under_model: 23
509 - Time for 1 entries: 1.01 s
517 - Time for 1 entries: 0.99 s
520 - Time for 1 entries: 1.01 s
508 - Time for 1 entries: 0.98 s
544 - Time for 1 entries: 0.76 s
Warning - min_possible_hfev1_under_model: 21
509 - Time for 1 entries: 0.91 s
520 - Time for 1 entries: 0.84 s
517 - Time for 1 entries: 1.05 s
544 - Time for 1 entries: 0.77 s
Warning - min_pos

/Users/tristan.trebaol/Desktop/DesktopMacTristan/PhD/Code/phd/src/inf_cutset_conditioning/cutset_cond_algs.py:874: RuntimeWarning: divide by zero encountered in log
  log_p_D_given_M = np.log(p_ecFEV1)


544 - Time for 1 entries: 0.66 s
Warning - min_possible_hfev1_under_model: 23
509 - Time for 1 entries: 0.84 s
520 - Time for 1 entries: 0.88 s
517 - Time for 1 entries: 0.88 s
508 - Time for 1 entries: 0.79 s
544 - Time for 1 entries: 0.68 s


/Users/tristan.trebaol/Desktop/DesktopMacTristan/PhD/Code/phd/src/inf_cutset_conditioning/cutset_cond_algs.py:874: RuntimeWarning: divide by zero encountered in log
  log_p_D_given_M = np.log(p_ecFEV1)


Warning - min_possible_hfev1_under_model: 22
509 - Time for 1 entries: 0.89 s
520 - Time for 1 entries: 0.85 s
517 - Time for 1 entries: 0.86 s
544 - Time for 1 entries: 0.72 s
508 - Time for 1 entries: 0.90 s
Warning - min_possible_hfev1_under_model: 21
509 - Time for 1 entries: 0.87 s
520 - Time for 1 entries: 0.87 s
517 - Time for 1 entries: 0.82 s
544 - Time for 1 entries: 0.65 s
Warning - min_possible_hfev1_under_model: 22
508 - Time for 1 entries: 0.74 s
520 - Time for 1 entries: 0.66 s
544 - Time for 1 entries: 0.57 s
517 - Time for 1 entries: 0.68 s
Warning - min_possible_hfev1_under_model: 21
508 - Time for 1 entries: 0.71 s
544 - Time for 1 entries: 0.50 s
Warning - min_possible_hfev1_under_model: 22
520 - Time for 1 entries: 0.77 s
517 - Time for 1 entries: 0.70 s
508 - Time for 1 entries: 0.72 s
544 - Time for 1 entries: 0.56 s
Warning - min_possible_hfev1_under_model: 23
520 - Time for 1 entries: 0.66 s
517 - Time for 1 entries: 0.74 s
508 - Time for 1 entries: 0.68 s
544 

In [7]:
res_fev1_fef_model

[-208.51498951380864,
 -233.5823006389506,
 -205.0249421323725,
 -218.15198953880153,
 -229.53473057420086,
 -209.897020878688,
 -230.11119038791244,
 -248.10503101552717,
 -234.89469924821242,
 -240.76382616906534,
 -207.0845895545836,
 -214.6687245863305,
 -237.78950733299496,
 -209.74283090612778,
 -233.51738992909688,
 -227.5527802299611,
 -219.765008607971,
 -230.6121842323716,
 -205.77849942059467,
 -223.60998174040665,
 -209.36830911190825,
 -245.77066647503307,
 -217.304116192268,
 -222.4828578955207,
 -251.90951327828856,
 -232.2891344433959,
 -204.150175521594,
 -210.19269063565127,
 -240.82291732252608,
 -220.8730547006791,
 -231.92849023787755,
 -251.38760752650853,
 -229.58750867233772,
 -203.94328512455155,
 -214.6375317989031,
 -333.5559414493162,
 -269.7548226293269,
 -205.73023895040436,
 -254.19013349756835,
 -222.1075001849066,
 -270.5592812608371,
 -212.5886390535689,
 -209.23525330407213,
 -214.41373386769408,
 -222.45878683789107,
 -229.5087393716263,
 -226.168660

In [9]:
a = np.argwhere(np.isnan(np.array(res_fev1_fef_model)))
print(a)
res_fev1_fef_model_2 = np.delete(res_fev1_fef_model, 49)
np.sum(res_fev1_fef_model_2)

[[49]]


-13344.58958026275

In [11]:
res_long_model_2 = np.delete(np.array(res_long_model), 49)
np.sum(res_long_model_2)

-9717.4197577834

In [16]:
res_2day_fev1_fef_model_2 = np.delete(np.array(res_2day_fev1_fef_model), 49)
np.sum(res_2day_fev1_fef_model_2)

-13533.923764966428

In [ ]:
import numpy as np
from scipy.stats import chi2

ll_full = -9717
ll_nested = -13345
# Degrees of freedom: difference in the number of parameters.
df_diff = 4455

# 1. Calculate the test statistic (D)
D = -2 * (ll_nested - ll_full)

# 2. Calculate the p-value from the chi-squared distribution
# The survival function (sf) is more accurate for extremely small p-values
p_value = chi2.sf(D, df_diff)

print(f"Test Statistic (D): {D:.4f}")
print(f"Degrees of Freedom: {df_diff}")
print(f"P-value: {p_value:.4g}") # .4g handles scientific notation

# The full model provides a statistically significant better fit, from the log-likelihood ratio test.

Test Statistic (D): 368.9119
Degrees of Freedom: 4455
P-value: 1


In [30]:
np.log(s1).sum()

-1520.4210354663614